# NYC Taxi Demand Forecasting — Data Ingestion

This notebook loads and validates raw NYC TLC Yellow Taxi trip records and the TLC taxi zone lookup before constructing the zone-hour demand dataset used for forecasting.

## Objective

Understand the structure, time coverage, and geographic identifiers in the raw trip data before making downstream aggregation or modeling decisions.


## 1. Imports

Load the Python libraries used for data ingestion and validation.


In [1]:
import pandas as pd

## 2. Load Raw Trip Data

Load the January 2025 Yellow Taxi trip records and inspect the dataset's dimensions and initial rows.


In [2]:
df = pd.read_parquet("../data/raw/taxi/yellow_tripdata_2025-01.parquet")

In [3]:
df.shape

(3475226, 20)

In [5]:
df.head()

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
0,1,2025-01-01 00:18:38,2025-01-01 00:26:59,1.0,1.60,1.0,N,229,237,1,10.0,3.5,0.5,3.00,0.0,1.0,18.00,2.5,0.0,0.0
1,1,2025-01-01 00:32:40,2025-01-01 00:35:13,1.0,0.50,1.0,N,236,237,1,5.1,3.5,0.5,2.02,0.0,1.0,12.12,2.5,0.0,0.0
2,1,2025-01-01 00:44:04,2025-01-01 00:46:01,1.0,0.60,1.0,N,141,141,1,5.1,3.5,0.5,2.00,0.0,1.0,12.10,2.5,0.0,0.0
3,2,2025-01-01 00:14:27,2025-01-01 00:20:01,3.0,0.52,1.0,N,244,244,2,7.2,1.0,0.5,0.00,0.0,1.0,9.70,0.0,0.0,0.0
4,2,2025-01-01 00:21:34,2025-01-01 00:25:06,3.0,0.66,1.0,N,244,116,2,5.8,1.0,0.5,0.00,0.0,1.0,8.30,0.0,0.0,0.0


## 3. Load Taxi Zone Lookup

Load the official taxi zone lookup used to translate pickup-location IDs into meaningful geographic zones. Inspect a known location ID to confirm the lookup structure.


In [7]:
zones = pd.read_csv("../data/raw/taxi/taxi_zone_lookup.csv")

In [9]:
zones.head()

,LocationID,Borough,Zone,service_zone
0,1,EWR,Newark Airport,EWR
1,2,Queens,Jamaica Bay,Boro Zone
2,3,Bronx,Allerton/Pelham Gardens,Boro Zone
3,4,Manhattan,Alphabet City,Yellow Zone
4,5,Staten Island,Arden Heights,Boro Zone


In [10]:
zones[zones["LocationID"] == 229]

,LocationID,Borough,Zone,service_zone
228,229,Manhattan,Sutton Place/Turtle Bay North,Yellow Zone


## 4. Restrict the Analysis to January 2025

Although the source file represents January 2025, validate pickup timestamps explicitly and retain only trips occurring from January 1 through January 31.


In [13]:
jan_mask = (
    (df["tpep_pickup_datetime"] >= "2025-01-01") &
    (df["tpep_pickup_datetime"] < "2025-02-01")
)

In [14]:
jan_mask.value_counts()

tpep_pickup_datetime
True     3475204
False         22
Name: count, dtype: int64

In [16]:
df = df[jan_mask].copy()
df.shape

/var/folders/cl/5ty07n953ks8b13dj78j5dmr0000gn/T/ipykernel_18828/1096253033.py:1: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  df = df[jan_mask].copy()


(3475204, 20)

**Observation:** The January source file contains a small number of pickup timestamps outside the intended January observation window. Filtering explicitly by pickup time ensures that downstream hourly demand calculations use a consistent calendar period.


## 5. Validate Pickup Location IDs

Check whether every pickup-location ID in the filtered trip data exists in the TLC taxi zone lookup before using those identifiers for spatial aggregation.


In [21]:
invalid_zones = ~df["PULocationID"].isin(zones["LocationID"])

invalid_zones.sum()

np.int64(0)

**Observation:** All pickup-location IDs in the filtered January trip data correspond to entries in the taxi zone lookup, so no trips need to be removed because of an invalid pickup-zone identifier.


## Next Step

The raw January trip records and pickup-location identifiers have been validated. The next stage is to aggregate individual trips into hourly pickup demand by taxi zone, explicitly represent zero-demand zone-hours, attach geographic information, and save the processed zone-hour panel.
